In [ ]:
import numpy as np
import os
import sys


os.chdir('/home/joeho/dphil/react_ml_test/')
sys.path.append('../')
from react_ml_test.helpers.utils import generate_error_table
from react_ml_test.helpers.model_eval import get_model_predictions, get_test_and_pred_data
from react_ml_test.helpers.io import get_name_from_path, load_data_cache
from react_ml_test.helpers.settings import CPD_MVK_DA_DATA_DIR, NPZ_DIR

test_set_fpaths = [f"{CPD_MVK_DA_DATA_DIR}/endo_DA_final_test_combined.xyz"]
test_set_names = [get_name_from_path(test_set_fpath) for test_set_fpath in test_set_fpaths]
model_names = ['GO-MACE-23', 'GO-MACE-23_endo_DA_n=8_r_dist_0_3_diverse_solv_25_fine_tuned_rep_1']

# test 1. check caches are different
npz_cache_1_fp = f"{NPZ_DIR}/GO-MACE-23_endo_DA_final_test_combined_energies_forces.npz"
npz_cache_2_fp = f"{NPZ_DIR}/GO-MACE-23_endo_DA_n=8_r_dist_0_3_diverse_solv_25_fine_tuned_rep_1_endo_DA_final_test_combined_energies_forces.npz"

cache_1_data = load_data_cache(npz_cache_1_fp)
energies_1, forces_1 = cache_1_data['energies'], cache_1_data['forces']

print("Cache 1 Data: ", cache_1_data)
print("Cache 1 Forces: ", forces_1)

cache_2_data = load_data_cache(npz_cache_2_fp)
energies_2, forces_2 = cache_2_data['energies'], cache_2_data['forces']

print("Cache 2 Data: ", cache_2_data)
print("Cache 2 Forces: ", forces_2)
assert not np.isclose(energies_1, energies_2).all()
assert not np.isclose(forces_1, forces_2).all()

# test 2:
# all_model_pred_energies, all_model_pred_forces = [], []
# for model_name in model_names:
#     model_pred_energies, model_pred_forces = get_model_predictions(model_name,
#                                                                    test_set_fpaths,
#                                                                    include_forces=True,
#                                                                    pbc_box=None)
#     print('Model Name: ', model_name)
#     print('Forces: ', model_pred_forces)
#     all_model_pred_energies.append(model_pred_energies)
#     all_model_pred_forces.append(model_pred_forces)


# test 3:
# get test set energies / forces and run inference on test set with models 
all_test_energies, all_test_forces, all_pred_energies_dict, all_pred_forces_dict = get_test_and_pred_data(model_names,
                                                                                                          test_set_fpaths)
print('All Pred Forces Dict: ', all_pred_forces_dict)

# test 4

# create rsme table of energies and forces for each model
energy_rmse_df = generate_error_table(all_pred_energies_dict, 
                                      all_test_energies, 
                                      test_set_names, 
                                      'rmse', 
                                      title="Energies (Shifted) RMSE (meV / atom)")
print('Energy RMSE DF: \n', energy_rmse_df)
forces_rmse_df = generate_error_table(all_pred_forces_dict, 
                                      all_test_forces, 
                                      test_set_names, 
                                      'rmse_comp', 
                                      title="Forces Comp. RMSE (meV / A)")
print('Forces RMSE DF: \n', forces_rmse_df)
